# 10 - Multi-Book Synthesis, Case Study, and Current State

Purpose:
- preserve the Gold signal interpretation layer from the original project;
- show how that same signal interacts with a **parallel multi-book evaluation**;
- keep one concrete historical case study and one current-state readout, then compare books side by side.

This notebook reads saved outputs from Notebooks 06–09 and does not recompute the Gold signal.


## Reader Orientation

The signal story is unchanged: Gold abnormality is a cross-market instability indicator. What changes here is the downstream question: **which books does that alarm help most, and how does the same Red or Amber state look across Brent, WTI, and Henry Hub proxy books?**

Prerequisite outputs: this notebook reads saved outputs from Steps 06, 08, and 09. Run those notebooks first in this project directory so that `gold_alarm_frame.parquet`, `riskbook_var_multi.parquet`, and the Step 09 comparison CSV files exist locally.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

ROOT = Path.cwd()
PROCESSED_DIR = ROOT / 'data' / 'processed'
OUTPUT_DIR = ROOT / 'outputs' / 'step10_synthesis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

alarm_frame = pd.read_parquet(PROCESSED_DIR / 'gold_alarm_frame.parquet')
signal_comps = pd.read_parquet(PROCESSED_DIR / 'gold_signal_components.parquet')
riskbook_multi = pd.read_parquet(PROCESSED_DIR / 'riskbook_var_multi.parquet')
book_lead = pd.read_csv(ROOT / 'outputs/step09_lead_time_dashboard/book_lead_time_comparison.csv')
dashboard_multi = pd.read_csv(ROOT / 'outputs/step09_lead_time_dashboard/dashboard_metrics_multi.csv', index_col=0, parse_dates=True)

print('Alarm frame:', alarm_frame.shape)
print('Signal comps:', signal_comps.shape)
print('Riskbook multi:', riskbook_multi.shape)
print('Dashboard multi:', dashboard_multi.shape)

## Section 1 — Regime Interpretation

The dashboard state is still driven by three signal families. Those interpretations do not depend on which downstream risk book is being reviewed.


In [ ]:
regime_map = pd.DataFrame([
    {'state': 'Amber', 'families_active': 'Relationship only',
     'stress_regime': 'Correlation breakdown — gold decoupling from usual drivers without a return spike.',
     'interpretation': 'Early structural regime shift. No return shock yet.',
     'action': 'Monitor. Inspect rolling correlation z-scores. No immediate VaR action.'},
    {'state': 'Amber', 'families_active': 'Return/vol only',
     'stress_regime': 'Gold own-shock without cross-signal corroboration. Possibly transient.',
     'interpretation': 'Single-family spike. Safe-haven surge or idiosyncratic gold move.',
     'action': 'Monitor. If residual or relationship joins within 5 days, escalate.'},
    {'state': 'Amber', 'families_active': 'Residual only',
     'stress_regime': 'Gold moving unexpectedly given current Brent/DXY/VIX/yield behaviour.',
     'interpretation': 'Something outside the conditioning variables is driving gold.',
     'action': 'Inspect signal components. Check for geopolitical events not in the model.'},
    {'state': 'Red', 'families_active': 'Return/vol + Relationship',
     'stress_regime': 'Safe-haven surge or liquidation dump coinciding with relationship regime shift.',
     'interpretation': 'Gold moving sharply while its market relationships restructure. Classic crisis onset.',
     'action': 'Run stress tests and review VaR calibration.'},
    {'state': 'Red', 'families_active': 'Return/vol + Residual',
     'stress_regime': 'Idiosyncratic gold shock not explained by current macro environment.',
     'interpretation': 'Gold is leading the market, not reacting to it. Cross-asset spillover risk elevated.',
     'action': 'Run stress tests and review VaR calibration.'},
    {'state': 'Red', 'families_active': 'Residual + Relationship',
     'stress_regime': 'Slow structural build — two corroborating signals without a sharp return spike yet.',
     'interpretation': 'Regime shift accumulating. VaR baseline may already be stale.',
     'action': 'Run stress tests and review VaR calibration.'},
    {'state': 'Red', 'families_active': 'All three families',
     'stress_regime': 'Full cross-market instability. All signal dimensions elevated simultaneously.',
     'interpretation': 'Highest-confidence stress signal. Multiple independent pathways confirming.',
     'action': 'Run stress tests and review VaR calibration immediately.'},
])
regime_map[['state', 'families_active', 'stress_regime', 'action']]

## Section 2 — COVID 2020 Case Study Across Books

The Gold alarm sequence is common. The downstream books are different. This section keeps one macro episode but shows how Brent, WTI, and Henry Hub proxy books react around the same alarm timing.


In [ ]:
CASE_START = '2020-02-01'
CASE_END = '2020-05-15'
FIRST_ALARM = pd.Timestamp('2020-02-25')

case_alarm = alarm_frame.loc[CASE_START:CASE_END]
case_comps = signal_comps.loc[CASE_START:CASE_END]
case_books = {
    book: riskbook_multi.loc[riskbook_multi['book'] == book].sort_index().loc[CASE_START:CASE_END]
    for book in sorted(riskbook_multi['book'].unique())
}

case_rows = []
for book_name, case_book in case_books.items():
    breach_days = case_book[case_book['var_breach'] == 1].index
    post_alarm_breaches = breach_days[breach_days >= FIRST_ALARM]
    first_breach = post_alarm_breaches[0] if len(post_alarm_breaches) > 0 else pd.NaT
    case_rows.append({
        'book': book_name,
        'first_post_alarm_breach': first_breach,
        'lead_days': (first_breach - FIRST_ALARM).days if pd.notna(first_breach) else np.nan,
        'var_breaches_in_window': int(len(breach_days)),
        'min_drawdown_in_window': float(case_book['drawdown'].min()) if len(case_book) else np.nan,
    })

case_study_summary = pd.DataFrame(case_rows)
case_study_summary

In [ ]:
STATE_COLOURS = {'Green': '#2ecc71', 'Amber': '#f39c12', 'Red': '#e74c3c'}
fig, axes = plt.subplots(len(case_books), 3, figsize=(15, 4 * len(case_books)), sharex=False)
if len(case_books) == 1:
    axes = np.array([axes])

for row_idx, (book_name, case_book) in enumerate(case_books.items()):
    rel_score = case_comps[[c for c in case_comps.columns if c.startswith('gold_corr_')]].abs().max(axis=1)

    axes[row_idx, 0].plot(case_comps.index, case_comps['gold_return_z'].abs(), linewidth=0.8, label='|return_z|')
    axes[row_idx, 0].plot(case_comps.index, case_comps['gold_vol_z'].clip(lower=0), linewidth=0.8, label='vol_z')
    axes[row_idx, 0].plot(case_comps.index, rel_score, linewidth=0.8, linestyle=':', label='max|corr_z|')
    axes[row_idx, 0].axhline(2.0, color='red', linestyle='--', linewidth=0.8)
    axes[row_idx, 0].axvline(FIRST_ALARM, color='black', linestyle='--', linewidth=1.0)
    axes[row_idx, 0].set_title(f'{book_name}: Gold signal families')
    axes[row_idx, 0].legend(fontsize=7)

    case_book[['R_book', 'hs_var_return']].plot(ax=axes[row_idx, 1], linewidth=0.8)
    breach_days = case_book.index[case_book['var_breach'] == 1]
    if len(breach_days):
        axes[row_idx, 1].scatter(breach_days, case_book.loc[breach_days, 'R_book'], color='red', s=14, label='VaR breach')
    axes[row_idx, 1].axvline(FIRST_ALARM, color='black', linestyle='--', linewidth=1.0)
    axes[row_idx, 1].set_title(f'{book_name}: Return vs VaR')
    axes[row_idx, 1].legend(fontsize=7)

    case_book['nav'].plot(ax=axes[row_idx, 2], linewidth=0.9)
    axes[row_idx, 2].axvline(FIRST_ALARM, color='black', linestyle='--', linewidth=1.0)
    axes[row_idx, 2].set_title(f'{book_name}: NAV through COVID window')

plt.tight_layout()

### Case Study Interpretation

The alarm timing is common across books because the Gold signal is common. The differences arise in how quickly each proxy book converts that regime shift into VaR breaches and drawdowns. This is the practical value of the **parallel multi-book evaluation**: one early-warning signal, several downstream exposures.


## Section 3 — Current State Across Books

In [ ]:
latest_signal_date = alarm_frame.index[-1]
latest_alarm = alarm_frame.loc[latest_signal_date]
current_rows = []
for book_name in sorted(riskbook_multi['book'].unique()):
    book_df = riskbook_multi.loc[riskbook_multi['book'] == book_name].sort_index()
    latest_book_date = book_df.index[-1]
    latest_book = book_df.iloc[-1]
    current_rows.append({
        'book': book_name,
        'signal_date': latest_signal_date,
        'book_date': latest_book_date,
        'dashboard_state': latest_alarm['dashboard_state'],
        'conditioned_score': int(latest_alarm['conditioned_alarm_score']),
        'latest_nav': latest_book['nav'],
        'latest_drawdown': latest_book['drawdown'],
        'latest_var_breach': int(latest_book['var_breach']),
        'latest_realized_vol_20d': latest_book['realized_vol_20d'],
    })

current_state_table = pd.DataFrame(current_rows)
current_state_table

## Final Claim

> Abnormal Gold behaviour remains a useful nowcasting escalation signal after the project is extended to multiple downstream books. The signal is built once, held fixed, and then tested against Brent futures, WTI Cushing spot, and Henry Hub spot proxy books under one common VaR and lead-time template. This preserves methodological discipline: the signal is not re-tuned for each market. Differences in coverage, lead time, and false-alarm burden across books therefore reflect differences in downstream exposure structure, not moving goalposts in the alarm itself. Weekly crude stocks and gas storage improve physical interpretation, but they remain context variables rather than P&L inputs, preventing look-ahead contamination.


In [ ]:
regime_map.to_csv(OUTPUT_DIR / 'regime_interpretation_table.csv', index=False)
case_study_summary.to_csv(OUTPUT_DIR / 'case_study_multi_book_summary.csv', index=False)
current_state_table.to_csv(OUTPUT_DIR / 'current_state_multi_book_table.csv', index=False)

print('Saved Step 10 multi-book synthesis outputs to:', OUTPUT_DIR)